# 01 — arXiv データ取得 & BERTopic 実験

ResearchTide v0.1 の最初のプロトタイプ。

1. arXiv API から cs.CL（計算言語学）の論文を取得
2. BERTopic でトピック抽出
3. 引用速度の時系列を可視化
4. Weak Signal 検出を試す

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

## 1. arXiv から論文を取得

In [ ]:
from researchtide.ingestion.arxiv import fetch_papers

# cs.CL の最新論文を 200 件取得（まずは小さく試す）
papers = fetch_papers(query="cat:cs.CL", max_results=200)
print(f"取得件数: {len(papers)}")
print(f"最新: {papers[0].title[:80]}...")
print(f"最古: {papers[-1].title[:80]}...")

In [ ]:
# データの中身を確認
import pandas as pd

df = pd.DataFrame([p.model_dump() for p in papers])
print(f"期間: {df['published'].min()} ~ {df['published'].max()}")
print(f"\nカテゴリ分布（上位10）:")
cats = df["categories"].explode().value_counts()
print(cats.head(10))

## 2. BERTopic でトピック抽出

In [ ]:
from researchtide.analysis.topics import extract_topics

# min_topic_size を小さめに（データが 200 件なので）
model, snapshots = extract_topics(papers, min_topic_size=5)

print(f"抽出トピック数: {len(snapshots)}")
print()
for s in snapshots[:10]:
    kw = ", ".join(s.keywords[:5])
    print(f"  Topic {s.topic_id}: {s.label} ({s.paper_count} papers)")
    print(f"    keywords: {kw}")

In [ ]:
# BERTopic の可視化（トピック間の距離マップ）
if model is not None:
    fig = model.visualize_topics()
    fig.show()

## 3. 引用速度の時系列

In [ ]:
from researchtide.analysis.citation_velocity import build_monthly_series, compute_acceleration

series = build_monthly_series(papers)
print(f"カテゴリ数: {len(series)}")

# cs.CL の月次推移を表示
if "cs.CL" in series:
    cl_series = series["cs.CL"]
    print(f"\ncs.CL 月次投稿数:")
    for month, count in cl_series[-12:]:
        bar = "█" * count
        print(f"  {month.strftime('%Y-%m')}: {count:3d} {bar}")
    
    acc = compute_acceleration(cl_series)
    print(f"\n加速度（直近3ヶ月平均）: {acc:.2f}")
    if acc > 0:
        print("→ 投稿ペースが加速中")
    elif acc < 0:
        print("→ 投稿ペースが減速中")
    else:
        print("→ 投稿ペースは安定")

## 4. Weak Signal 検出

トピックの citation_velocity を仮で設定して Isolation Forest を試す。
（S2 API で引用データを補完すれば実データになる）

In [ ]:
import random
from researchtide.detection.weak_signal import detect_weak_signals

# 仮の citation_velocity を paper_count ベースで生成
# （実運用では S2 API の引用データから計算する）
for s in snapshots:
    s.citation_velocity = s.paper_count * random.uniform(0.5, 3.0)

detected = detect_weak_signals(snapshots)

print("Weak Signal / Rising 検出結果:")
for s in detected:
    if s.status.value in ("weak_signal", "rising"):
        print(f"  [{s.status.value:12s}] {s.label} (velocity={s.citation_velocity:.1f}, papers={s.paper_count})")

## 5. 影響伝播グラフ（プレビュー）

引用データがないため、ここではグラフ構築のインターフェースだけ確認する。
S2 API で references を補完すれば実グラフが生成される。

In [ ]:
from researchtide.graph.influence import build_influence_graph

# トピック割り当て（BERTopic の結果から）
if model is not None:
    docs = [p.abstract for p in papers if p.abstract]
    topics_assigned = model.transform(docs)[0]
    
    papers_with_abstract = [p for p in papers if p.abstract]
    topic_assignments = {
        p.paper_id: t
        for p, t in zip(papers_with_abstract, topics_assigned)
        if t != -1
    }
    
    topic_labels = {s.topic_id: s.label for s in snapshots}
    G = build_influence_graph(papers, topic_assignments, topic_labels)
    
    print(f"グラフ: {G.number_of_nodes()} ノード, {G.number_of_edges()} エッジ")
    print("（引用データなしのため、エッジは 0 の想定。S2 enrichment 後に再実行）")

---

## 次のステップ

- [ ] Semantic Scholar API で引用データを補完 → 実際の citation_velocity を計算
- [ ] 取得件数を増やす（200 → 2000+）、複数カテゴリを跨ぐ
- [ ] 影響伝播グラフに実データを入れて可視化（PyVis）
- [ ] 時系列を遡って、過去のトレンド変化を検証